In [ ]:
import os
import tempfile
import subprocess
import boto3

In [31]:
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_ENDPOINT_URL = os.getenv("AWS_ENDPOINT_URL")
AWS_DEFAULT_REGION = os.getenv("AWS_DEFAULT_REGION")

if not AWS_ACCESS_KEY_ID:
    raise ValueError("Please set AWS_ACCESS_KEY_ID in the .env file")
if not AWS_SECRET_ACCESS_KEY:
    raise ValueError("Please set AWS_SECRET_ACCESS_KEY in the .env file")
if not AWS_DEFAULT_REGION:
    raise ValueError("Please set AWS_DEFAULT_REGION in the .env file")

In [32]:
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_DEFAULT_REGION
)

In [28]:
bucket = "aitraf"
prefix = "raw/"

In [29]:
paginator = s3.get_paginator("list_objects_v2")
keys = []
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for o in page.get("Contents", []):
        k = o["Key"]
        if k.lower().endswith(".mov"):
            keys.append(k)

for k in keys:
    with tempfile.TemporaryDirectory() as d:
        mov = os.path.join(d, "in.mov")
        mp4 = os.path.join(d, "out.mp4")

        s3.download_file(bucket, k, mov)
        subprocess.run([
            "ffmpeg", "-y",
            "-i", mov,
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-r", "30",
            "-c:a", "aac",
            "-movflags", "+faststart",
            mp4
        ], check=True)

        new_key = os.path.splitext(k)[0] + ".mp4"
        s3.upload_file(mp4, bucket, new_key, ExtraArgs={"ContentType": "video/mp4"})

        s3.delete_object(Bucket=bucket, Key=k)

print(f"Replaced {len(keys)} .mov files with .mp4 in s3://{bucket}/{prefix}")

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

Replaced 23 .mov files with .mp4 in s3://aitraf/raw/
